In [ ]:
!pip install numpy pandas matplotlib scipy yfinance arch statsmodels -q

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

# Chart style
BLUE = '#1A3A6E'; RED = '#DC3545'; GREEN = '#2E7D32'
ORANGE = '#E67E22'; GRAY = '#666666'; PURPLE = '#8E44AD'

plt.rcParams.update({
    'figure.facecolor': 'none', 'axes.facecolor': 'none',
    'savefig.facecolor': 'none', 'savefig.transparent': True,
    'axes.grid': False, 'font.size': 10, 'axes.labelsize': 11,
    'axes.titlesize': 12, 'legend.fontsize': 9, 'xtick.labelsize': 9,
    'ytick.labelsize': 9, 'axes.spines.top': False, 'axes.spines.right': False,
    'lines.linewidth': 1.0, 'axes.linewidth': 0.6,
    'legend.facecolor': 'none', 'legend.framealpha': 0, 'legend.edgecolor': 'none',
    'figure.dpi': 150,
})

def save_chart(fig, name):
    fig.savefig(f'{name}.pdf', bbox_inches='tight', transparent=True, dpi=150)
    fig.savefig(f'{name}.png', bbox_inches='tight', transparent=True, dpi=150)
    print(f'Saved: {name}')

def legend_bottom(ax, ncol=2, y=-0.18):
    ax.legend(loc='upper center', bbox_to_anchor=(0.5, y), ncol=ncol, frameon=False)

In [ ]:
# ============================================================
# Nelson (1990) Theorem: IGARCH strict stationarity
# ============================================================
# IGARCH: σ²_t = ω + α ε²_{t-1} + (1-α) σ²_{t-1}
# Strict stationarity requires E[ln(α z² + (1-α))] < 0

from scipy.integrate import quad

def lyapunov_exponent(alpha):
    """Compute E[ln(α z² + (1-α))] for z ~ N(0,1)"""
    integrand = lambda z: np.log(alpha * z**2 + (1 - alpha)) * stats.norm.pdf(z)
    result, _ = quad(integrand, -10, 10)
    return result

alphas = np.linspace(0.001, 0.999, 200)
lyap = [lyapunov_exponent(a) for a in alphas]

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(alphas, lyap, color=BLUE, lw=2)
ax.axhline(0, color='black', lw=0.5, ls='-')
ax.fill_between(alphas, lyap, 0, where=[l < 0 for l in lyap],
                alpha=0.15, color=GREEN, label='Strictly stationary (γ < 0)')
ax.fill_between(alphas, lyap, 0, where=[l >= 0 for l in lyap],
                alpha=0.15, color=RED, label='Non-stationary (γ ≥ 0)')

# Find critical alpha
crit_idx = np.argmin(np.abs(lyap))
crit_alpha = alphas[crit_idx]
ax.axvline(crit_alpha, color=ORANGE, lw=1.5, ls='--',
           label=f'Critical α ≈ {crit_alpha:.3f}')

ax.set_xlabel('α (IGARCH parameter)')
ax.set_ylabel('Lyapunov exponent E[ln(αz² + 1-α)]')
ax.set_title('Nelson (1990): IGARCH Strict Stationarity Region', fontweight='bold')
legend_bottom(ax, ncol=3, y=-0.15)
save_chart(fig, 'garch_nelson_theorem')
plt.show()

print(f"Critical α ≈ {crit_alpha:.4f}")
print(f"For α < {crit_alpha:.3f}: IGARCH is strictly stationary despite α+β=1")
print(f"Typical GARCH α ≈ 0.05-0.15: well inside the stationary region")

In [ ]:
# Simulation: IGARCH paths with different ω
np.random.seed(42)
T = 2000
omega_vals = [0.01, 0.05, 0.10]
alpha_ig = 0.10

fig, axes = plt.subplots(len(omega_vals), 1, figsize=(12, 8), sharex=True)
for ax, omega in zip(axes, omega_vals):
    sigma2 = np.zeros(T)
    eps = np.zeros(T)
    sigma2[0] = 1.0
    for t in range(1, T):
        z = np.random.standard_normal()
        eps[t] = np.sqrt(sigma2[t-1]) * z
        sigma2[t] = omega + alpha_ig * eps[t-1]**2 + (1 - alpha_ig) * sigma2[t-1]

    ax.plot(np.sqrt(sigma2), color=BLUE, lw=0.5)
    ax.set_ylabel('σ_t')
    ax.set_title(f'IGARCH(1,1): ω={omega}, α={alpha_ig}, β={1-alpha_ig}', fontsize=10)

axes[-1].set_xlabel('Time')
plt.suptitle('IGARCH Conditional Volatility Paths', fontweight='bold', y=1.01)
plt.tight_layout()
save_chart(fig, 'garch_igarch_paths')
plt.show()